# 03 · Matrix (2-D Arrays)

A matrix is a list of lists. `matrix[i][j]` is the element in **row `i`, column `j`**.

**Topics covered**
1. Traversal and pattern printing (triangles, diagonals)
2. Transpose
3. Set matrix zeroes
4. Rotate matrix by 90°
5. Spiral traversal

**Handy index rules for an `n × n` matrix**

| Region | Condition |
|---|---|
| Main diagonal | `i == j` |
| Upper triangle (incl. diagonal) | `j >= i` |
| Lower triangle (incl. diagonal) | `i >= j` |
| Anti-diagonal | `i + j == n - 1` |

## 1. Traversal and patterns

Two nested loops: the outer loop walks the rows, the inner loop walks the columns. `print(..., end=" ")` keeps a row on one line, and the bare `print()` moves to the next line.

In [1]:
nums = [[5, 2, 3],
        [4, 6, 7],
        [1, 67, 3]]
rows = len(nums)
columns = len(nums[0])

print("Full matrix:")
for i in range(rows):
    for j in range(columns):
        print(nums[i][j], end=" ")
    print()

Full matrix:
5 2 3 
4 6 7 
1 67 3 


In [2]:
print("Upper triangle (j >= i):")
for i in range(rows):
    for j in range(columns):
        print(nums[i][j] if j >= i else "*", end=" ")
    print()

print("\nLower triangle (i >= j):")
for i in range(rows):
    for j in range(columns):
        print(nums[i][j] if i >= j else "*", end=" ")
    print()

Upper triangle (j >= i):
5 2 3 
* 6 7 
* * 3 

Lower triangle (i >= j):
5 * * 
4 6 * 
1 67 3 


In [3]:
print("Main diagonal (i == j):")
for i in range(rows):
    for j in range(columns):
        print(nums[i][j] if i == j else "*", end=" ")
    print()

print("\nAnti-diagonal (i + j == rows - 1):")
for i in range(rows):
    for j in range(columns):
        print(nums[i][j] if i + j == rows - 1 else "*", end=" ")
    print()

print("\nTop-left corner (i + j <= 1):")
for i in range(rows):
    for j in range(columns):
        print(nums[i][j] if i + j <= 1 else "*", end=" ")
    print()

Main diagonal (i == j):
5 * * 
* 6 * 
* * 3 

Anti-diagonal (i + j == rows - 1):
* * 3 
* 6 * 
1 * * 

Top-left corner (i + j <= 1):
5 2 * 
4 * * 
* * * 


## 2. Transpose of a matrix

Transpose = flip the matrix over its main diagonal: the element at `[i][j]` moves to `[j][i]`. Rows become columns.

For an `rows × cols` matrix the result has shape `cols × rows`.

**Time:** O(rows·cols) · **Space:** O(rows·cols)

> 🐞 **Fix:** the earlier version assigned `row = len(nums)` but used `rows` below, which only worked by accident because `rows` was defined in an earlier cell.

In [4]:
rows = len(nums)
cols = len(nums[0])
result = [[0] * rows for _ in range(cols)]   # note the swapped dimensions
for i in range(rows):
    for j in range(cols):
        result[j][i] = nums[i][j]
print(result)                                # [[5, 4, 1], [2, 6, 67], [3, 7, 3]]

[[5, 4, 1], [2, 6, 67], [3, 7, 3]]


## 3. Set matrix zeroes

If an element is `0`, set its **entire row and column** to `0`.

**Approach 1 – mark with a sentinel (infinity):** for every `0`, turn the other non-zero cells in its row/column into `inf`; at the end convert every `inf` to `0`. We can't mark them `0` immediately, otherwise new zeros would trigger more zeroing.
Time O(n·m·(n+m)), Space O(1).

**Approach 2 – row & column trackers:** two small arrays remember which rows/columns contain a zero.
Time O(n·m), Space O(n+m).

*(A space-O(1) version exists that uses the first row and first column of the matrix itself as the trackers.)*

In [5]:
# Approach 1: infinity markers
class Solution:
    def mark_infinity(self, matrix, row, col):
        r, c = len(matrix), len(matrix[0])
        for i in range(r):                       # whole column
            if matrix[i][col] != 0:
                matrix[i][col] = float("inf")
        for j in range(c):                       # whole row
            if matrix[row][j] != 0:
                matrix[row][j] = float("inf")

    def set_zeroes(self, matrix):
        r, c = len(matrix), len(matrix[0])
        for i in range(r):
            for j in range(c):
                if matrix[i][j] == 0:
                    self.mark_infinity(matrix, i, j)
        for i in range(r):
            for j in range(c):
                if matrix[i][j] == float("inf"):
                    matrix[i][j] = 0

matrix = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
Solution().set_zeroes(matrix)
for row in matrix:
    print(row)

[0, 0, 0, 0]
[0, 4, 5, 0]
[0, 3, 1, 0]


In [6]:
# Approach 2: row/column trackers
matrix = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
r, c = len(matrix), len(matrix[0])
row_has_zero = [False] * r
col_has_zero = [False] * c

for i in range(r):                               # pass 1: remember where zeros are
    for j in range(c):
        if matrix[i][j] == 0:
            row_has_zero[i] = True
            col_has_zero[j] = True

for i in range(r):                               # pass 2: zero out marked rows/columns
    for j in range(c):
        if row_has_zero[i] or col_has_zero[j]:
            matrix[i][j] = 0

for row in matrix:
    print(row)

[0, 0, 0, 0]
[0, 4, 5, 0]
[0, 3, 1, 0]


## 4. Rotate a matrix by 90° (clockwise)

### Method 1 – extra matrix
After a clockwise rotation, the element at `[i][j]` ends up at `[j][n-1-i]`.
Time O(n²), Space O(n²).

### Method 2 – in place: **transpose, then reverse each row** ✅
```
[1 2 3]      transpose      [1 4 7]     reverse rows     [7 4 1]
[4 5 6]    ──────────▶      [2 5 8]   ──────────────▶    [8 5 2]
[7 8 9]                     [3 6 9]                      [9 6 3]
```
Time O(n²), Space O(1).

In [7]:
# Method 1: new matrix
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
n = len(matrix)
result = [[0] * n for _ in range(n)]
for i in range(n):
    for j in range(n):
        result[j][n - 1 - i] = matrix[i][j]
print("Extra matrix:", result)

Extra matrix: [[7, 4, 1], [8, 5, 2], [9, 6, 3]]


In [8]:
# Method 2: in place
def rotate(matrix):
    n = len(matrix)
    for i in range(n - 1):                       # transpose: swap across the diagonal
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:                           # reverse every row
        row.reverse()

matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate(matrix)
print("In place    :", matrix)

In place    : [[7, 4, 1], [8, 5, 2], [9, 6, 3]]


### Bonus: can matrix `mat` become `target` by rotating?
Rotate up to **4 times** (0°, 90°, 180°, 270°) and compare after each rotation.

> 🐞 **Fix:** the earlier `rotate` function had this check pasted *inside* it, using variables `mat` and `target` that weren't defined. It's now a separate function.

In [9]:
import copy

def find_rotation(mat, target):
    mat = copy.deepcopy(mat)                     # don't destroy the caller's matrix
    for _ in range(4):
        if mat == target:
            return True
        rotate(mat)
    return False

print(find_rotation([[0, 1], [1, 0]], [[1, 0], [0, 1]]))   # True  (one 90° turn)
print(find_rotation([[0, 1], [1, 1]], [[1, 0], [0, 1]]))   # False

True
False


## 5. Spiral traversal

Walk the outer ring clockwise, then shrink the boundaries and repeat. Four boundaries: `top`, `bottom`, `left`, `right`.

1. **Left → right** along the `top` row, then `top += 1`
2. **Top → bottom** along the `right` column, then `right -= 1`
3. **Right → left** along the `bottom` row (only if rows remain), then `bottom -= 1`
4. **Bottom → top** along the `left` column (only if columns remain), then `left += 1`

The `if` checks before steps 3 and 4 stop us from re-reading a row/column when the matrix is not square.

**Time:** O(n·m) · **Space:** O(1) extra

In [10]:
def spiral(matrix):
    top, left = 0, 0
    bottom, right = len(matrix) - 1, len(matrix[0]) - 1
    result = []
    while top <= bottom and left <= right:
        for i in range(left, right + 1):             # → along top row
            result.append(matrix[top][i])
        top += 1
        for i in range(top, bottom + 1):             # ↓ down right column
            result.append(matrix[i][right])
        right -= 1
        if top <= bottom:                            # ← along bottom row
            for i in range(right, left - 1, -1):
                result.append(matrix[bottom][i])
            bottom -= 1
        if left <= right:                            # ↑ up left column
            for i in range(bottom, top - 1, -1):
                result.append(matrix[i][left])
            left += 1
    return result

print(spiral([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))              # [1, 2, 3, 6, 9, 8, 7, 4, 5]
print(spiral([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))  # non-square works too

[1, 2, 3, 6, 9, 8, 7, 4, 5]
[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


---
### ✅ Key takeaways
- Memorise the index conditions: `i == j`, `i + j == n - 1`, `j >= i`.
- **Rotate 90° = transpose + reverse each row.**
- Use boundary variables (`top, bottom, left, right`) for spiral-type traversals.